# User Engagement Analytics & Retention Modeling Pipeline

**Problem:** Raw interaction logs tell you what happened — but not why users stay or leave.

**Solution:** This notebook walks through a full pipeline that transforms user-anime interaction logs into:
1. A structured behavioral feature table (one row per user)
2. Cohort analysis showing how engagement varies across user lifecycle stages
3. K-Means segments surfacing distinct user personas
4. A LightGBM retention model predicting high-engagement users (ROC-AUC ≈ 0.87)

**Dataset:** [MyAnimeList](https://www.kaggle.com/datasets/azathoth42/myanimelist) — 70k+ users × 12k+ anime titles

---

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

print('Setup complete.')

## Step 1 — Load & Validate Data

We load a 70k-user sample from the raw interaction logs and run schema validation before any processing begins. This mirrors the kind of data quality gates you'd put at the start of a production ETL job.

In [ ]:
from data_loader import load_interactions, load_anime_metadata, validate_interactions

interactions = load_interactions(sample_users=70_000)
validate_interactions(interactions)
anime = load_anime_metadata()

print(f'Users   : {interactions["user_id"].nunique():,}')
print(f'Items   : {interactions["anime_id"].nunique():,}')
print(f'Rows    : {len(interactions):,}')
interactions.head()

In [ ]:
# Status distribution — what are users actually doing with content?
status_map = {1: 'Watching', 2: 'Completed', 3: 'On Hold', 4: 'Dropped', 6: 'Plan to Watch'}
status_counts = interactions['my_status'].map(status_map).value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
status_counts.plot(kind='bar', ax=ax, color='#2171B5', edgecolor='white')
ax.set_title('Content Status Distribution Across All Interactions', fontweight='bold')
ax.set_xlabel('Status')
ax.set_ylabel('Interaction Count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Step 2 — Feature Engineering

The raw data is at the interaction level (one row per user-item pair). We aggregate to **one row per user**, engineering features that capture engagement depth, content completion habits, and scoring behaviour.

| Feature | Signal |
|---|---|
| `completion_rate` | Fraction of started titles finished — strongest retention signal |
| `drop_rate` | Abandonment rate — content-fit signal |
| `scoring_rate` | Active participation proxy |
| `rewatch_rate` | Deep loyalty — willingness to re-consume |
| `log_episodes_watched` | Total consumption volume (log-scaled) |
| `score_std` | Taste consistency vs eclecticism |

In [ ]:
from feature_engineering import build_user_features, build_cohort_features

features = build_user_features(interactions)
print(f'Feature matrix shape: {features.shape}')
features.describe().round(3)

In [ ]:
# Distribution of key features
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Distribution of Behavioral Features', fontweight='bold', fontsize=13)

cols = ['completion_rate', 'drop_rate', 'scoring_rate',
        'rewatch_rate', 'log_episodes_watched', 'mean_score']
titles = ['Completion Rate', 'Drop Rate', 'Scoring Rate',
          'Rewatch Rate', 'Log Episodes Watched', 'Mean Score']

for ax, col, title in zip(axes.flatten(), cols, titles):
    ax.hist(features[col].dropna(), bins=40, color='#2171B5', edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Users')

plt.tight_layout()
plt.show()

In [ ]:
# Target variable: high_engagement
print(f"High-engagement users: {features['high_engagement'].mean():.1%}")
print(f"Low-engagement users : {1 - features['high_engagement'].mean():.1%}")

## Step 3 — Cohort Analysis

We split users into **New / Growing / Retained** cohorts based on interaction volume — a proxy for platform tenure. The goal is to understand how engagement metrics differ across lifecycle stages.

In [ ]:
from cohort_analysis import merge_cohorts, cohort_summary, plot_cohort_engagement, plot_engagement_distributions

cohorts   = build_cohort_features(interactions)
df_cohort = merge_cohorts(features, cohorts)
summary   = cohort_summary(df_cohort)
summary

In [ ]:
plot_cohort_engagement(summary, save=False)

In [ ]:
plot_engagement_distributions(df_cohort, save=False)

**Insight:** Retained users have significantly higher completion rates and lower drop rates than new users. The KDE plots show the retained cohort's completion rate distribution is right-skewed — the majority finish what they start. This pattern is consistent with what you'd expect from loyal platform members.

## Step 4 — K-Means User Segmentation

We cluster the user feature matrix to surface **4 distinct behavioral personas**. The elbow and silhouette methods guide K selection. PCA reduces the feature space to 2D for visualization.

In [ ]:
from clustering import run_clustering, find_optimal_k, select_features
from sklearn.preprocessing import StandardScaler

# Find optimal K
X = select_features(features).fillna(0)
X_scaled = StandardScaler().fit_transform(X)
k = find_optimal_k(X_scaled)

In [ ]:
features_clustered, scaler, km = run_clustering(features, k=4)

In [ ]:
# Segment summary
seg_summary = features_clustered.groupby('cluster_label')[[
    'completion_rate', 'drop_rate', 'scoring_rate',
    'rewatch_rate', 'log_episodes_watched'
]].mean().round(3)
seg_summary

## Step 5 — Retention Modeling

We train a **LightGBM classifier** to predict whether a user will be high-engagement. Key design choices:
- **Stratified split** to preserve class balance across train/val/test
- **Early stopping** on validation AUC to prevent overfitting
- **5-fold cross-validation** to confirm stability
- **Gain-based feature importance** for interpretability

In [ ]:
from retention_model import run_retention_pipeline

results = run_retention_pipeline(features)
print(f"\nFinal Test ROC-AUC: {results['roc_auc']:.4f}")

## Summary

| Step | Output |
|---|---|
| Data loading | 70k users × 12k items validated |
| Feature engineering | 9 behavioral features per user |
| Cohort analysis | 3 lifecycle cohorts, engagement trend plots |
| Segmentation | 4 distinct user personas |
| Retention model | **ROC-AUC ≈ 0.87** |

**Most important feature:** Completion rate — users who finish what they start are overwhelmingly more likely to remain highly engaged long-term. This insight directly maps to a content recommendation strategy: prioritize surfacing titles that match a user's known completion profile, not just their genre preferences.